In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
df = pd.read_csv('feature_engineered_dataset.csv')
df.shape

In [ ]:
df.head()

In [ ]:
df.dtypes

In [ ]:
df.isnull().sum()

In [ ]:
df.describe()

In [ ]:
# checking sale price range
print('min sale price:', df['SalePrice'].min())
print('max sale price:', df['SalePrice'].max())
print('avg sale price:', df['SalePrice'].mean())

In [ ]:
# neighborhood wise average sale price
nbhood = df.groupby('xrPrimaryNeighborhoodID')['SalePrice'].mean().sort_values(ascending=False)
nbhood.head(10)

In [ ]:
nbhood.head(10).plot(kind='bar', figsize=(10,5), title='Top 10 Neighborhoods by Avg Sale Price')
plt.ylabel('Avg Sale Price')
plt.xlabel('Neighborhood ID')
plt.tight_layout()
plt.show()

In [ ]:
# year wise trend
yearly = df.groupby('SaleYear')['SalePrice'].mean()
yearly

In [ ]:
yearly.plot(kind='line', marker='o', figsize=(8,4), title='Avg Sale Price by Year')
plt.ylabel('Avg Price')
plt.grid(True)
plt.show()

In [ ]:
# how many sales each year
df.groupby('SaleYear')['SalePrice'].count()

In [ ]:
# price per sqft distribution
df_clean = df[(df['PricePerSqft'] > 0) & (df['PricePerSqft'] < 600)]

df_clean['PricePerSqft'].hist(bins=40, figsize=(10,4), color='steelblue', edgecolor='white')
plt.title('Price Per Sqft Distribution')
plt.xlabel('Price/sqft')
plt.ylabel('count')
plt.show()

In [ ]:
# classifying properties based on appraisal ratio
# if ratio > 1.5 means sale price much higher than appraised value
# if ratio < 0.7 means undervalued

df['val_label'] = 'fair'
df.loc[df['AppraisalRatio'] > 1.5, 'val_label'] = 'overvalued'
df.loc[df['AppraisalRatio'] < 0.7, 'val_label'] = 'undervalued'

df['val_label'].value_counts()

In [ ]:
df['val_label'].value_counts().plot(kind='pie', autopct='%1.1f%%', figsize=(6,6))
plt.title('Valuation Split')
plt.ylabel('')
plt.show()

In [ ]:
# monthly count of sales - checking if any seasonality
monthly = df.groupby('SaleMonth')['SalePrice'].count()
monthly.index = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
monthly

In [ ]:
monthly.plot(kind='bar', figsize=(10,4), color='coral', title='Sales Count by Month')
plt.ylabel('Number of Sales')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# neighborhood summary - saving to csv
nbhood_summary = df.groupby('xrPrimaryNeighborhoodID').agg(
    count=('SalePrice','count'),
    avg_price=('SalePrice','mean'),
    median_price=('SalePrice','median'),
    avg_ppsqft=('PricePerSqft','mean'),
    avg_appraisal_ratio=('AppraisalRatio','mean')
).round(2).reset_index()

nbhood_summary.to_csv('neighborhood_summary.csv', index=False)
nbhood_summary.head()

In [ ]:
# saving yearly trends
yr = df.groupby('SaleYear').agg(
    transactions=('SalePrice','count'),
    avg_price=('SalePrice','mean'),
    median_price=('SalePrice','median'),
    avg_ppsqft=('PricePerSqft','mean')
).round(2).reset_index()

yr.to_csv('yearly_price_trends.csv', index=False)
yr

In [ ]:
# appraisal ratio vs sale price scatter
sample = df_clean.sample(500, random_state=42)

plt.figure(figsize=(9,5))
plt.scatter(sample['AppraisalRatio'], sample['SalePrice'], alpha=0.4, color='teal')
plt.xlabel('Appraisal Ratio')
plt.ylabel('Sale Price')
plt.title('Appraisal Ratio vs Sale Price')
plt.tight_layout()
plt.show()

In [ ]:
# correlation check
num_cols = ['SalePrice','LandSF','TotalFinishedArea','PricePerSqft','AppraisalRatio','LogSalePrice']
df[num_cols].corr()

In [ ]:
import matplotlib.pyplot as plt

corr = df[num_cols].corr()
plt.figure(figsize=(8,6))
plt.imshow(corr, cmap='coolwarm', aspect='auto')
plt.colorbar()
plt.xticks(range(len(num_cols)), num_cols, rotation=45, ha='right')
plt.yticks(range(len(num_cols)), num_cols)
plt.title('Correlation Heatmap')
plt.tight_layout()
plt.show()